# T/NK Cell Subcluster Annotation + scANVI Retrain
Version: v1.0 (2026-03-18)

**Purpose**:
  1. Load trained reference h5ad (X_scvi / X_scanvi already present)
  2. For each target L2 cell type, compute within-type neighbors on X_scvi
  3. Run Leiden at multiple resolutions and identify markers
  4. Save per-cluster marker CSVs and UMAP plots for annotation review
  5. --- USER FILLS ANNOTATION DICTS IN PART 3 ---
  6. Apply annotations -> build new scanvi_label column
  7. Reload scVI model -> SCANVI.from_scvi_model with new labels -> train
  8. Save refined reference model + h5ad

**Input**:
  - adata_tnk_scanvi_ref_20260315_v1_2.h5ad  (post-scVI/scANVI, HVG subset + .raw full gene)
  - tnk_scvi_ref_model/                      (trained scVI, needed for scANVI rebuild)

**Output**:
  - subcluster_markers/<L2_type>/markers_leiden_rX.X.csv
  - subcluster_markers/<L2_type>/umap_leiden_rX.X.pdf
  - adata_tnk_scanvi_ref_retrain_v1_0.h5ad
  - tnk_scvi_ref_model_retrain/
  - tnk_scanvi_ref_model_retrain/

**QRM rules applied**:
  - QRM basis='umap' (NOT 'X_umap') in sc.pl.embedding
  - QRM sc.settings.vector_friendly=True before plot blocks; no rasterized= kwarg
  - QRM use_raw=True for rank_genes_groups (full gene .raw)
  - QRM 15.1 : reference training uses from_scvi_model (correct here)
  - QRM 15.4 : covariates on full gene space (already done in prior step)
  - QRM 13   : category dtype before write_h5ad

## 0. Configuration

In [ ]:
# ============================================================================
# CONFIGURATION  (edit here only)
# ============================================================================

INPUT_H5AD       = "/home/h2048/data/py/0315/tnk_scarches_ref/adata_tnk_scanvi_ref_20260315_v1_2.h5ad"
SCVI_MODEL_DIR   = "/home/h2048/data/py/0315/tnk_scarches_ref/tnk_scvi_ref_model"
OUTPUT_DIR       = "/home/h2048/data/py/0318/tnk_subcluster_retrain"

# Column to use for splitting into subgroups for per-type subclustering
SPLIT_KEY        = "cell_type_L2"   # NK cells / CD4 T cells / CD8 T cells

# Cell types to subcluster (set to None to auto-detect from SPLIT_KEY)
TARGET_CELL_TYPES = None   # e.g. ['NK cells', 'CD4 T cells', 'CD8 T cells']

# Latent representation to use for within-type subclustering
# X_scanvi: scANVI-refined latent with label supervision (recommended for annotated data)
# X_scvi: batch-corrected VAE latent (use if X_scanvi not available)
SUBCLUSTER_REP   = "X_scanvi"

# Leiden resolutions to test per cell type
LEIDEN_RESOLUTIONS = [0.3, 0.5, 0.8, 1.0]

# Resolution to use for annotation (can override per cell type in Part 3)
DEFAULT_ANNOTATION_RESOLUTION = 0.5

# Batch key (for scVI reload)
BATCH_KEY        = "sample"

# Existing scANVI labels key in input h5ad (for reference during annotation)
OLD_LABELS_KEY   = "scanvi_label"

# New fine-grained scANVI labels key (output)
NEW_LABELS_KEY   = "scanvi_label_refined"
UNLABELED        = "Unknown"

# scANVI retrain parameters
SCANVI_EPOCHS    = 200
BATCH_SIZE       = 256
N_SAMPLES_PER_LABEL = None   # None = auto (80% of min class size, capped at 100)

# Marker detection parameters
MARKER_METHOD    = "wilcoxon"
N_TOP_MARKERS    = 20        # top markers per cluster in CSV
MIN_IN_GROUP_FRACTION = 0.1  # minimum fraction of cells expressing in group

# Visualization
DPI              = 300
SEED             = 42

print("Configuration loaded")
print(f"  Input h5ad   : {INPUT_H5AD}")
print(f"  scVI model   : {SCVI_MODEL_DIR}")
print(f"  Output       : {OUTPUT_DIR}")

## 1. Imports & Setup

In [ ]:
import os, gc, warnings
warnings.filterwarnings("ignore")

os.environ["OMP_NUM_THREADS"] = "8"
os.environ["MKL_NUM_THREADS"] = "8"
os.environ["OPENBLAS_NUM_THREADS"] = "8"

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import numpy as np
import pandas as pd
import scipy.sparse as sparse
import scanpy as sc
import scvi
from pathlib import Path

sc.settings.verbosity  = 2
sc.settings.n_jobs     = 16
sc.settings.set_figure_params(dpi=DPI, facecolor="white", frameon=False)

np.random.seed(SEED)
scvi.settings.seed = SEED

output_dir   = Path(OUTPUT_DIR)
marker_dir   = output_dir / "subcluster_markers"
fig_dir      = output_dir / "figures"
for d in [output_dir, marker_dir, fig_dir]:
    d.mkdir(parents=True, exist_ok=True)
sc.settings.figdir = str(fig_dir)

print(f"scanpy : {sc.__version__}")
print(f"scvi   : {scvi.__version__}")
print(f"Output : {output_dir}")

## 2. Load Data

In [ ]:
print("Loading h5ad...")
adata = sc.read_h5ad(INPUT_H5AD)
print(f"  Shape  : {adata.shape}")
print(f"  obsm   : {list(adata.obsm.keys())}")
print(f"  layers : {list(adata.layers.keys()) if adata.layers else 'None'}")
print(f"  .raw   : {adata.raw is not None}  "
      f"({adata.raw.n_vars} genes)" if adata.raw is not None else "  .raw : None")

# Verify required keys
assert SUBCLUSTER_REP in adata.obsm, \
    f"'{SUBCLUSTER_REP}' not found in obsm -- run reference training first"
assert SPLIT_KEY in adata.obs.columns, \
    f"'{SPLIT_KEY}' not found in obs"

# Determine cell types to subcluster
if TARGET_CELL_TYPES is None:
    target_cell_types = sorted(adata.obs[SPLIT_KEY].dropna().unique().tolist())
else:
    target_cell_types = TARGET_CELL_TYPES

print(f"\nCell types to subcluster ({SPLIT_KEY}): {target_cell_types}")
for ct in target_cell_types:
    n = (adata.obs[SPLIT_KEY] == ct).sum()
    print(f"  {ct}: {n:,} cells")

## 3. Per-Cell-Type Subclustering on X_scvi Space

For each L2 cell type:
  1. Extract subset (keeps existing .raw reference)
  2. Compute subset-specific neighbors on SUBCLUSTER_REP
  3. Run Leiden at multiple resolutions
  4. Compute markers (use_raw=True for full gene space)
  5. Save CSVs and UMAPs for inspection
  6. Store chosen clustering back to main adata.obs

In [ ]:
# Storage: per-cell-type Leiden assignments stored in main adata.obs
# Key pattern: leiden_sub_{sanitized_celltype}_r{resolution}
subcluster_summary = {}  # {cell_type: {'n_clusters': int, 'res_used': float, ...}}

def sanitize_colname(s):
    """Convert cell type name to safe column name."""
    return s.lower().replace(" ", "_").replace("/", "_").replace("+", "plus")

for cell_type in target_cell_types:
    print(f"\n{'='*60}")
    print(f"Subclustering: {cell_type}")
    print(f"{'='*60}")

    # --- Extract subset ---
    mask    = adata.obs[SPLIT_KEY] == cell_type
    adata_s = adata[mask].copy()
    n_cells = adata_s.n_obs
    print(f"  Cells: {n_cells:,}")

    if n_cells < 50:
        print(f"  [SKIP] < 50 cells -- skipping subclustering for {cell_type}")
        continue

    ct_safe  = sanitize_colname(cell_type)
    ct_mdir  = marker_dir / ct_safe
    ct_mdir.mkdir(exist_ok=True)

    # Transfer the latent representation slice
    rep_matrix = adata_s.obsm[SUBCLUSTER_REP]

    # --- Compute within-subset neighbors on latent space ---
    # n_neighbors capped at min(30, n_cells//5) to avoid errors on small subsets
    n_neighbors = min(30, max(5, n_cells // 5))
    print(f"  Computing neighbors on {SUBCLUSTER_REP} (n_neighbors={n_neighbors})...")
    sc.pp.neighbors(adata_s, use_rep=SUBCLUSTER_REP, n_neighbors=n_neighbors)

    # --- UMAP (re-compute within subset for better visualization) ---
    sc.tl.umap(adata_s, min_dist=0.3, spread=1.0)

    # --- Multi-resolution Leiden ---
    res_results = {}
    for res in LEIDEN_RESOLUTIONS:
        key = f"leiden_r{res}"
        try:
            sc.tl.leiden(adata_s, resolution=res, key_added=key)
            n_clust = adata_s.obs[key].nunique()
            res_results[res] = n_clust
            print(f"  Leiden r={res}: {n_clust} clusters")
        except Exception as e:
            print(f"  Leiden r={res} failed: {e}")

    if not res_results:
        print(f"  [ERROR] All Leiden runs failed for {cell_type} -- skipping")
        continue

    # --- Marker genes for each resolution ---
    for res in LEIDEN_RESOLUTIONS:
        key = f"leiden_r{res}"
        if key not in adata_s.obs.columns:
            continue

        n_clust = res_results.get(res, 0)
        if n_clust <= 1:
            print(f"  Skipping markers for r={res} (only {n_clust} cluster)")
            continue

        print(f"  Computing markers for r={res} ({n_clust} clusters)...")
        try:
            sc.tl.rank_genes_groups(
                adata_s,
                groupby  = key,
                method   = MARKER_METHOD,
                use_raw  = True,
                pts      = True,   # compute pct expressing
                key_added= f"rgg_{key}",
            )

            # Extract top markers per cluster
            all_markers = []
            for clust in adata_s.obs[key].cat.categories:
                df = sc.get.rank_genes_groups_df(
                    adata_s,
                    group     = clust,
                    key       = f"rgg_{key}",
                    pval_cutoff = 0.05,
                    log2fc_min  = 0.25,
                )
                if df.empty:
                    continue
                # Filter by minimum in-group fraction
                if "pct_nz_group" in df.columns:
                    df = df[df["pct_nz_group"] >= MIN_IN_GROUP_FRACTION]
                df.insert(0, "cluster", clust)
                df.insert(1, "cell_type", cell_type)
                df.insert(2, "resolution", res)
                all_markers.append(df.head(N_TOP_MARKERS))

            if all_markers:
                markers_df = pd.concat(all_markers, ignore_index=True)
                csv_path   = ct_mdir / f"markers_leiden_r{res}.csv"
                markers_df.to_csv(csv_path, index=False)
                print(f"    Saved: {csv_path.name}")
            else:
                print(f"    No significant markers found for r={res}")

        except Exception as e:
            print(f"    Markers r={res} failed: {e}")

    # --- UMAP plots for each resolution ---
    # QRM: set vector_friendly BEFORE plot block; do NOT pass rasterized= to embedding
    sc.settings.vector_friendly = True

    n_res_valid = sum(1 for r in LEIDEN_RESOLUTIONS if f"leiden_r{r}" in adata_s.obs.columns)
    ncols = min(n_res_valid + 2, 4)
    nrows = int(np.ceil((n_res_valid + 2) / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5, nrows * 4.5))
    axes_flat = np.array(axes).ravel()
    ax_idx    = 0

    # Existing L3 label (ground truth reference)
    if "cell_type_L3" in adata_s.obs.columns:
        sc.pl.embedding(adata_s, basis="umap", color="cell_type_L3",
                        title="L3 (reference labels)",
                        ax=axes_flat[ax_idx], show=False,
                        legend_loc="right margin", legend_fontsize=6)
        ax_idx += 1

    # Old scANVI prediction
    if OLD_LABELS_KEY in adata_s.obs.columns:
        sc.pl.embedding(adata_s, basis="umap", color=OLD_LABELS_KEY,
                        title=f"Old scANVI ({OLD_LABELS_KEY})",
                        ax=axes_flat[ax_idx], show=False,
                        legend_loc="right margin", legend_fontsize=6)
        ax_idx += 1

    for res in LEIDEN_RESOLUTIONS:
        key = f"leiden_r{res}"
        if key not in adata_s.obs.columns:
            continue
        sc.pl.embedding(adata_s, basis="umap", color=key,
                        title=f"Leiden r={res} ({res_results.get(res,'?')} clusters)",
                        ax=axes_flat[ax_idx], show=False,
                        legend_loc="on data", legend_fontsize=7)
        ax_idx += 1

    for i in range(ax_idx, len(axes_flat)):
        axes_flat[i].set_visible(False)

    plt.suptitle(f"{cell_type} -- Subclustering Overview", fontsize=13, y=1.01)
    plt.tight_layout()
    fig.savefig(ct_mdir / "umap_resolutions_overview.pdf",
                bbox_inches="tight", dpi=DPI)
    plt.close(fig)
    print(f"  Saved: {ct_mdir}/umap_resolutions_overview.pdf")

    # --- Dotplot of top markers at chosen resolution ---
    chosen_key = f"leiden_r{DEFAULT_ANNOTATION_RESOLUTION}"
    if chosen_key in adata_s.obs.columns:
        rgg_key = f"rgg_{chosen_key}"
        if rgg_key in adata_s.uns:
            try:
                sc.settings.vector_friendly = True
                dp = sc.pl.rank_genes_groups_dotplot(
                    adata_s,
                    n_genes  = 5,
                    key      = rgg_key,
                    groupby  = chosen_key,
                    use_raw  = True,
                    show     = False,
                    return_fig = True,
                )
                dp.savefig(ct_mdir / f"dotplot_r{DEFAULT_ANNOTATION_RESOLUTION}.pdf",
                           bbox_inches="tight", dpi=DPI)
                plt.close()
                print(f"  Saved: dotplot_r{DEFAULT_ANNOTATION_RESOLUTION}.pdf")
            except Exception as e:
                print(f"  Dotplot failed: {e}")

    # --- Store chosen-resolution clusters back in main adata ---
    # Use DEFAULT_ANNOTATION_RESOLUTION; override per cell type in Part 3 if needed
    anno_key = f"leiden_r{DEFAULT_ANNOTATION_RESOLUTION}"
    if anno_key in adata_s.obs.columns:
        main_col  = f"subcluster_{ct_safe}"
        # Prefix cluster IDs with cell type safe name to avoid cross-type collision
        adata.obs.loc[mask, main_col] = (
            adata_s.obs[anno_key].astype(str)
            .apply(lambda x: f"{ct_safe}_{x}").values
        )
        n_sub = adata_s.obs[anno_key].nunique()
        subcluster_summary[cell_type] = {
            "n_cells"            : n_cells,
            "annotation_key"     : main_col,
            "annotation_res"     : DEFAULT_ANNOTATION_RESOLUTION,
            "n_subclusters"      : n_sub,
            "resolutions_tested" : LEIDEN_RESOLUTIONS,
        }
        print(f"  Stored in adata.obs['{main_col}']: {n_sub} subclusters")

    del adata_s
    gc.collect()

print("\n" + "="*60)
print("Subclustering complete. Summary:")
for ct, info in subcluster_summary.items():
    print(f"  {ct}: {info['n_subclusters']} subclusters -> obs['{info['annotation_key']}']")
print(f"\nReview marker CSVs in: {marker_dir}")
print("Then fill ANNOTATION_MAP dicts in Part 3 below.")

## 4. Print Annotation Templates

Run this cell AFTER Part 3 subclustering.
Copy the output into the ANNOTATION_MAP dicts in Part 4.

In [ ]:
print("="*70)
print("ANNOTATION TEMPLATES -- copy cluster IDs from your marker review")
print("="*70)

for cell_type, info in subcluster_summary.items():
    ct_safe    = sanitize_colname(cell_type)
    anno_key   = info["annotation_key"]
    res        = info["annotation_res"]

    print(f"\n# --- {cell_type} (obs['{anno_key}'], r={res}) ---")
    print(f"ANNOTATION_MAP_{ct_safe} = {{")

    if anno_key in adata.obs.columns:
        clusters = sorted(adata.obs[anno_key].dropna().unique().tolist())
        for c in clusters:
            print(f'    "{c}": "",   # fill in label')
    else:
        print(f"    # WARNING: '{anno_key}' not found in obs -- check Part 3 log")

    print("}")

print("\n# After filling labels above, paste into Part 4 'USER FILLS' section.")
print(f"\nMarker CSVs for each cell type are in: {marker_dir}")

## 5. USER FILLS ANNOTATION MAPS

=============================================================================
INSTRUCTIONS:
  1. Check marker CSVs in subcluster_markers/<cell_type>/markers_leiden_rX.X.csv
  2. Check UMAPs in subcluster_markers/<cell_type>/umap_resolutions_overview.pdf
  3. For each cell type, fill in the empty "" labels in the dicts below
  4. Optionally override resolution per cell type via ANNOTATION_RES_OVERRIDE
  5. Run this cell, then continue to Part 5
=============================================================================

In [ ]:
# ============================================================================
# USER FILLS: Replace "" with your annotation label for each cluster
# Cluster IDs follow the pattern: {cell_type_safe}_{cluster_number}
# Example: "nk_cells_0": "NK Cytotoxic"
# Use UNLABELED = "Unknown" for clusters you want to exclude from supervision
# ============================================================================

# NK cells
ANNOTATION_MAP_nk_cells = {
    # "nk_cells_0": "NK Cytotoxic",
    # "nk_cells_1": "NK Resting",
    # "nk_cells_2": "NK IFN-high",
    # fill in after reviewing marker CSVs
}

# CD4 T cells
ANNOTATION_MAP_cd4_t_cells = {
    # "cd4_t_cells_0": "CD4 Naive/TCM",
    # "cd4_t_cells_1": "CD4 TEM",
    # fill in after reviewing marker CSVs
}

# CD8 T cells
ANNOTATION_MAP_cd8_t_cells = {
    # "cd8_t_cells_0": "CD8 TRM",
    # "cd8_t_cells_1": "CD8 TEMRA",
    # fill in after reviewing marker CSVs
}

# ============================================================================
# Optional: Override annotation resolution per cell type
# If a cell type requires a different resolution than DEFAULT_ANNOTATION_RESOLUTION
# Example: {"NK cells": 0.8, "CD4 T cells": 0.5}
# ============================================================================
ANNOTATION_RES_OVERRIDE = {}

# ============================================================================
# Combine all annotation maps (keys = subcluster IDs in adata.obs columns)
# If you add a new cell type, add its map here as well
# ============================================================================
COMBINED_ANNOTATION_MAP = {}
COMBINED_ANNOTATION_MAP.update(ANNOTATION_MAP_nk_cells)
COMBINED_ANNOTATION_MAP.update(ANNOTATION_MAP_cd4_t_cells)
COMBINED_ANNOTATION_MAP.update(ANNOTATION_MAP_cd8_t_cells)

# Count filled labels
n_filled   = sum(1 for v in COMBINED_ANNOTATION_MAP.values() if v.strip() != "")
n_total    = len(COMBINED_ANNOTATION_MAP)
n_unlabeled_entries = sum(1 for v in COMBINED_ANNOTATION_MAP.values() if v == UNLABELED)

print(f"Annotation map: {n_filled}/{n_total} entries filled")
print(f"  '{UNLABELED}' entries (supervised exclusions): {n_unlabeled_entries}")
if n_filled == 0:
    print("[WARNING] No labels filled -- Part 5 will assign all cells to UNLABELED")
    print("  scANVI will still train but without supervision")

## 6. Apply Annotations -> Build NEW_LABELS_KEY column

Re-runs Part 3 subclustering for any cell types with ANNOTATION_RES_OVERRIDE,
then assembles the new scanvi_label column from all annotation maps.

In [ ]:
# If resolution overrides require re-running subclustering for affected types,
# reload subset, run leiden at override res, re-store in obs

for cell_type, override_res in ANNOTATION_RES_OVERRIDE.items():
    if cell_type not in subcluster_summary:
        print(f"[SKIP] {cell_type} not in subcluster_summary -- run Part 3 first")
        continue

    prev_res = subcluster_summary[cell_type]["annotation_res"]
    if override_res == prev_res:
        continue

    print(f"Re-running Leiden at override res={override_res} for {cell_type}...")
    mask    = adata.obs[SPLIT_KEY] == cell_type
    adata_s = adata[mask].copy()
    ct_safe = sanitize_colname(cell_type)

    n_neighbors = min(30, max(5, adata_s.n_obs // 5))
    sc.pp.neighbors(adata_s, use_rep=SUBCLUSTER_REP, n_neighbors=n_neighbors)

    sc.tl.leiden(adata_s, resolution=override_res,
                 key_added=f"leiden_r{override_res}")

    main_col = f"subcluster_{ct_safe}"
    adata.obs.loc[mask, main_col] = (
        adata_s.obs[f"leiden_r{override_res}"].astype(str)
        .apply(lambda x: f"{ct_safe}_{x}").values
    )
    subcluster_summary[cell_type]["annotation_res"] = override_res
    subcluster_summary[cell_type]["annotation_key"] = main_col
    n_sub = adata_s.obs[f"leiden_r{override_res}"].nunique()
    subcluster_summary[cell_type]["n_subclusters"]  = n_sub
    print(f"  Updated: {n_sub} subclusters at r={override_res}")

    del adata_s
    gc.collect()

# --- Build NEW_LABELS_KEY from all annotation maps ---
print(f"\nBuilding '{NEW_LABELS_KEY}' column...")
adata.obs[NEW_LABELS_KEY] = UNLABELED  # start with all Unknown

n_assigned = 0
n_missing  = 0

for cell_type, info in subcluster_summary.items():
    anno_key = info["annotation_key"]
    ct_safe  = sanitize_colname(cell_type)

    if anno_key not in adata.obs.columns:
        print(f"  [WARN] '{anno_key}' not in obs -- {cell_type} all Unknown")
        continue

    for cluster_id, label in COMBINED_ANNOTATION_MAP.items():
        if not cluster_id.startswith(ct_safe + "_"):
            continue
        if label.strip() == "":
            label = UNLABELED   # treat empty as unlabeled

        mask = adata.obs[anno_key] == cluster_id
        n    = int(mask.sum())
        if n == 0:
            print(f"  [WARN] Cluster '{cluster_id}' not found in obs['{anno_key}']")
            n_missing += 1
            continue

        adata.obs.loc[mask, NEW_LABELS_KEY] = label
        n_assigned += n

print(f"\nAssignment summary:")
print(f"  Total cells      : {adata.n_obs:,}")
print(f"  Labeled (non-Unknown): {(adata.obs[NEW_LABELS_KEY] != UNLABELED).sum():,} ({n_assigned:,} via map)")
print(f"  Unknown          : {(adata.obs[NEW_LABELS_KEY] == UNLABELED).sum():,}")
print(f"  Unresolved map keys: {n_missing}")
print(f"\nLabel distribution:")
print(adata.obs[NEW_LABELS_KEY].value_counts())

# Convert to category
adata.obs[NEW_LABELS_KEY] = adata.obs[NEW_LABELS_KEY].astype("category")
if UNLABELED not in adata.obs[NEW_LABELS_KEY].cat.categories:
    adata.obs[NEW_LABELS_KEY] = adata.obs[NEW_LABELS_KEY].cat.add_categories([UNLABELED])

## 7. UMAP Overview with New Labels (Sanity Check Before Retrain)

In [ ]:
sc.settings.vector_friendly = True

fig, axes = plt.subplots(1, 3, figsize=(21, 6))

# Check if X_umap exists on main adata; if not, compute from X_scanvi
if "X_umap" not in adata.obsm:
    print("X_umap not found in main adata -- computing from X_scanvi...")
    sc.pp.neighbors(adata, use_rep="X_scanvi", n_neighbors=30)
    sc.tl.umap(adata, min_dist=0.3)

sc.pl.embedding(adata, basis="umap",
                color=OLD_LABELS_KEY if OLD_LABELS_KEY in adata.obs.columns else SPLIT_KEY,
                title="Before: Old scANVI Labels",
                ax=axes[0], show=False,
                legend_loc="right margin", legend_fontsize=7)

sc.pl.embedding(adata, basis="umap", color=NEW_LABELS_KEY,
                title=f"After: New Labels ({NEW_LABELS_KEY})",
                ax=axes[1], show=False,
                legend_loc="right margin", legend_fontsize=7)

sc.pl.embedding(adata, basis="umap", color=SPLIT_KEY,
                title=f"L2 Cell Type ({SPLIT_KEY})",
                ax=axes[2], show=False,
                legend_loc="right margin", legend_fontsize=7)

plt.suptitle("Annotation Sanity Check Before scANVI Retrain", fontsize=13, y=1.01)
plt.tight_layout()
fig.savefig(fig_dir / "annotation_sanity_check.pdf",
            bbox_inches="tight", dpi=DPI)
plt.close(fig)
print(f"Saved: annotation_sanity_check.pdf")
print("\nVerify this plot before proceeding to retrain.")
print(f"If labels look wrong, edit Part 5 and re-run Parts 5-6.")

## 8. Reload scVI Model + Train scANVI with New Labels

Key design:
  - scVI was already trained on HVG subset; we reload it (no retraining scVI).
  - adata loaded here IS the HVG subset (the input h5ad is post-scVI output).
  - SCANVI.from_scvi_model: correct for reference training (QRM 15.1).
  - weight_decay=0.0 is not required for from_scvi_model but kept for consistency.

In [ ]:
import torch

print("Loading scVI model for scANVI rebuild...")
print(f"  scVI model dir : {SCVI_MODEL_DIR}")
print(f"  Input adata    : {adata.shape} (HVG subset)")

# Verify adata has 'counts' layer (required by scVI model)
if "counts" not in adata.layers:
    raise ValueError(
        "'counts' layer not found in adata. "
        "Expected HVG-subset h5ad with counts + log1p layers."
    )

# Ensure NEW_LABELS_KEY is category with UNLABELED present
if UNLABELED not in adata.obs[NEW_LABELS_KEY].cat.categories:
    adata.obs[NEW_LABELS_KEY] = adata.obs[NEW_LABELS_KEY].cat.add_categories([UNLABELED])

# Prepare label distribution for adaptive n_samples_per_label
label_counts = adata.obs[NEW_LABELS_KEY].value_counts()
labeled_only = label_counts[label_counts.index != UNLABELED]
min_class_size = int(labeled_only.min()) if len(labeled_only) > 0 else 10

if N_SAMPLES_PER_LABEL is None:
    n_samples_per_label = max(10, min(100, int(min_class_size * 0.8)))
else:
    n_samples_per_label = N_SAMPLES_PER_LABEL

print(f"\nscANVI label stats:")
print(f"  Labeled classes : {len(labeled_only)}")
print(f"  Smallest class  : '{labeled_only.idxmin()}' ({min_class_size} cells)")
print(f"  n_samples_per_label (adaptive): {n_samples_per_label}")
print(f"\nAll label counts:")
print(label_counts)

In [ ]:
# Reload scVI from saved model dir
# Pass adata directly -- model var_names must match adata.var_names
vae = scvi.model.SCVI.load(SCVI_MODEL_DIR, adata=adata)
print(f"\nscVI model loaded")
print(f"  n_latent : {vae.module.n_latent}")

# Verify latent representation consistency (X_scvi should match loaded model)
# Get one batch of latent from reloaded model; compare mean to existing X_scvi
test_latent = vae.get_latent_representation(indices=list(range(min(500, adata.n_obs))))
existing    = adata.obsm[SUBCLUSTER_REP][:min(500, adata.n_obs)]
corr        = float(np.corrcoef(test_latent.ravel(), existing.ravel())[0, 1])
print(f"  X_scvi correlation (reloaded vs stored): {corr:.4f}")
if corr < 0.95:
    print(f"  [WARN] Low correlation ({corr:.3f}) -- model may have been retrained "
          f"with different data. Proceeding but verify output.")

In [ ]:
# Build scANVI from reloaded scVI
print("\nBuilding scANVI from reloaded scVI...")

lvae = scvi.model.SCANVI.from_scvi_model(
    vae,
    unlabeled_category = UNLABELED,
    labels_key         = NEW_LABELS_KEY,
)

n_labeled   = int((adata.obs[NEW_LABELS_KEY] != UNLABELED).sum())
n_unlabeled = int((adata.obs[NEW_LABELS_KEY] == UNLABELED).sum())
print(f"  labels_key      : {NEW_LABELS_KEY}")
print(f"  labeled cells   : {n_labeled:,} ({n_labeled/adata.n_obs*100:.1f}%)")
print(f"  unknown cells   : {n_unlabeled:,} ({n_unlabeled/adata.n_obs*100:.1f}%)")
label_cats = sorted(
    [c for c in adata.obs[NEW_LABELS_KEY].cat.categories if c != UNLABELED]
)
print(f"  supervision classes ({len(label_cats)}): {label_cats}")

In [ ]:
print(f"\nTraining scANVI (max_epochs={SCANVI_EPOCHS}, batch_size={BATCH_SIZE})...")
lvae.train(
    max_epochs              = SCANVI_EPOCHS,
    batch_size              = BATCH_SIZE,
    train_size              = 0.9,
    early_stopping          = True,
    early_stopping_patience = 20,
    n_samples_per_label     = n_samples_per_label,
)
print("scANVI training complete")

In [ ]:
# Extract outputs
adata.obsm["X_scanvi_refined"]        = lvae.get_latent_representation()
adata.obs["scanvi_pred_refined"]       = lvae.predict()
scanvi_proba                           = lvae.predict(soft=True)
adata.obs["scanvi_pred_prob_refined"]  = scanvi_proba.max(axis=1).values.astype(np.float32)

print(f"X_scanvi_refined shape : {adata.obsm['X_scanvi_refined'].shape}")
concordance = (
    adata.obs["scanvi_pred_refined"].astype(str) ==
    adata.obs[NEW_LABELS_KEY].astype(str)
).mean()
print(f"Overall concordance (pred vs label): {concordance:.3f}")
print(f"\nPrediction confidence:\n{adata.obs['scanvi_pred_prob_refined'].describe()}")

low_conf = (adata.obs["scanvi_pred_prob_refined"] < 0.5).sum()
print(f"Low-confidence cells (<0.5): {low_conf:,} ({low_conf/adata.n_obs*100:.1f}%)")

# Per-class concordance
if n_labeled > 0:
    print("\nPer-class concordance:")
    for cls in sorted(adata.obs[NEW_LABELS_KEY].cat.categories):
        if cls == UNLABELED:
            continue
        mask = adata.obs[NEW_LABELS_KEY].astype(str) == cls
        if mask.sum() == 0:
            continue
        acc = (adata.obs.loc[mask, "scanvi_pred_refined"].astype(str) == cls).mean()
        print(f"  {cls:<35}: {acc:.3f}  (n={mask.sum():,})")

In [ ]:
# Plot training loss curve BEFORE del lvae
try:
    fig_loss, ax_loss = plt.subplots(figsize=(8, 4))
    train_hist = lvae.history.get("train_loss_epoch", None)
    if train_hist is not None:
        ax_loss.plot(train_hist.values.flatten(), label="train ELBO")
    ax_loss.set_xlabel("Epoch"); ax_loss.set_ylabel("ELBO"); ax_loss.legend()
    ax_loss.set_title("scANVI Retrain Training Loss")
    fig_loss.savefig(fig_dir / "scanvi_retrain_training_loss.pdf",
                     bbox_inches="tight", dpi=DPI)
    plt.close(fig_loss)
    print("Saved: scanvi_retrain_training_loss.pdf")
except Exception as e:
    print(f"  Training loss plot failed: {e}")

In [ ]:
# Save scANVI model
scanvi_model_dir_new = str(output_dir / "tnk_scanvi_ref_model_retrain")
lvae.save(scanvi_model_dir_new, overwrite=True)
pd.Series(adata.var_names.tolist()).to_csv(
    Path(scanvi_model_dir_new) / "var_names.csv", index=False, header=False
)
print(f"scANVI model saved: {scanvi_model_dir_new}/")

del vae, lvae
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("GPU memory released")

## 9. Post-Retrain Visualization

In [ ]:
# Recompute neighbors + UMAP on new X_scanvi_refined for final visualization
print("Computing neighbors on X_scanvi_refined for final UMAP...")
sc.pp.neighbors(adata, use_rep="X_scanvi_refined", n_neighbors=30)
sc.tl.umap(adata, min_dist=0.3, spread=1.0, key_added="X_umap_refined")

In [ ]:
sc.settings.vector_friendly = True

fig, axes = plt.subplots(2, 3, figsize=(24, 14))

sc.pl.embedding(adata, basis="umap_refined", color=NEW_LABELS_KEY,
                title=f"New Labels ({NEW_LABELS_KEY})",
                ax=axes[0, 0], show=False,
                legend_loc="right margin", legend_fontsize=6)

sc.pl.embedding(adata, basis="umap_refined", color="scanvi_pred_refined",
                title="scANVI Refined Predictions",
                ax=axes[0, 1], show=False,
                legend_loc="right margin", legend_fontsize=6)

sc.pl.embedding(adata, basis="umap_refined", color="scanvi_pred_prob_refined",
                title="Prediction Confidence (new)",
                ax=axes[0, 2], show=False, color_map="RdYlGn", vmin=0, vmax=1)

sc.pl.embedding(adata, basis="umap_refined", color=SPLIT_KEY,
                title=f"L2 Cell Type ({SPLIT_KEY})",
                ax=axes[1, 0], show=False,
                legend_loc="right margin", legend_fontsize=7)

sc.pl.embedding(adata, basis="umap_refined", color=BATCH_KEY,
                title=f"Batch ({BATCH_KEY})",
                ax=axes[1, 1], show=False,
                legend_loc="right margin", legend_fontsize=5)

# Concordance map: green = correct prediction, red = mismatch
concordance_col = (
    adata.obs["scanvi_pred_refined"].astype(str) ==
    adata.obs[NEW_LABELS_KEY].astype(str)
).map({True: "correct", False: "mismatch"}).astype("category")
adata.obs["_pred_concordance"] = concordance_col
sc.pl.embedding(adata, basis="umap_refined", color="_pred_concordance",
                title="Prediction Concordance",
                ax=axes[1, 2], show=False,
                palette={"correct": "#2ecc71", "mismatch": "#e74c3c"})

plt.suptitle("Post-Retrain Overview -- scANVI Refined", fontsize=14, y=1.01)
plt.tight_layout()
fig.savefig(fig_dir / "umap_post_retrain_overview.pdf",
            bbox_inches="tight", dpi=DPI)
plt.close(fig)
print("Saved: umap_post_retrain_overview.pdf")

In [ ]:
# Key marker feature plots on refined UMAP
MARKER_GENES = [
    "CD3D", "CD4", "CD8A", "GNLY", "NKG7", "FCGR3A",
    "CCR7", "SELL", "TCF7",
    "GZMB", "PRF1", "GZMK",
    "ITGA1", "CXCR6", "CD69",
    "PDCD1", "HAVCR2", "FOXP3",
    "KLRG1", "CX3CR1",
    "KIT", "ZBTB16", "TRDC",
]
avail_raw    = set(adata.raw.var_names) if adata.raw is not None else set(adata.var_names)
marker_valid = [g for g in MARKER_GENES if g in avail_raw]
missing_mks  = [g for g in MARKER_GENES if g not in avail_raw]
if missing_mks:
    print(f"Markers not in .raw: {missing_mks}")

sc.settings.vector_friendly = True

ncols = 5
nrows = int(np.ceil(len(marker_valid) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 3.5))
axes_flat  = axes.ravel()

for i, gene in enumerate(marker_valid):
    sc.pl.embedding(adata, basis="umap_refined", color=gene,
                    ax=axes_flat[i], show=False,
                    use_raw=(adata.raw is not None),
                    color_map="viridis", vmin=0)
    axes_flat[i].set_title(gene, fontsize=10, fontweight="bold")

for i in range(len(marker_valid), len(axes_flat)):
    axes_flat[i].set_visible(False)

plt.suptitle("T/NK Key Markers -- Refined UMAP (use_raw=True)", fontsize=13, y=1.01)
plt.tight_layout()
fig.savefig(fig_dir / "featureplot_refined_umap.pdf",
            bbox_inches="tight", dpi=DPI)
plt.close(fig)
print("Saved: featureplot_refined_umap.pdf")

## 10. uns Metadata + Cleanup + Save

In [ ]:
import anndata as _anndata
_anndata.settings.allow_write_nullable_strings = True

# Remove temp concordance column
if "_pred_concordance" in adata.obs.columns:
    adata.obs.drop(columns=["_pred_concordance"], inplace=True)

# Write uns metadata
adata.uns["retrain_params"] = {
    "script_version"       : "v1.0",
    "base_input_h5ad"      : INPUT_H5AD,
    "scvi_model_dir"       : SCVI_MODEL_DIR,
    "scanvi_model_dir_new" : scanvi_model_dir_new,
    "split_key"            : SPLIT_KEY,
    "subcluster_rep"       : SUBCLUSTER_REP,
    "default_anno_res"     : DEFAULT_ANNOTATION_RESOLUTION,
    "leiden_resolutions"   : LEIDEN_RESOLUTIONS,
    "old_labels_key"       : OLD_LABELS_KEY,
    "new_labels_key"       : NEW_LABELS_KEY,
    "unlabeled_category"   : UNLABELED,
    "scanvi_epochs"        : SCANVI_EPOCHS,
    "n_samples_per_label"  : n_samples_per_label,
    "batch_key"            : BATCH_KEY,
    "scanvi_label_categories": sorted([
        c for c in adata.obs[NEW_LABELS_KEY].cat.categories if c != UNLABELED
    ]),
    "subcluster_summary"   : {
        ct: {k: v for k, v in info.items() if k != "resolutions_tested"}
        for ct, info in subcluster_summary.items()
    },
    "combined_annotation_map": COMBINED_ANNOTATION_MAP,
}

# Convert all label columns to category dtype before write
cat_cols = [NEW_LABELS_KEY, "scanvi_pred_refined", SPLIT_KEY, BATCH_KEY,
            OLD_LABELS_KEY if OLD_LABELS_KEY in adata.obs.columns else None]
for col in cat_cols:
    if col and col in adata.obs.columns:
        adata.obs[col] = adata.obs[col].astype("category")

# Rename any reserved '_index' column
for attr in ("obs", "var"):
    df = getattr(adata, attr)
    if "_index" in df.columns:
        setattr(adata, attr, df.rename(columns={"_index": "orig_index"}))

In [ ]:
out_h5ad = output_dir / "adata_tnk_scanvi_ref_retrain_v1_0.h5ad"
print(f"Writing {out_h5ad.name}...")
print(f"  .X shape      : {adata.shape}")
print(f"  .raw shape    : {adata.raw.n_obs} x {adata.raw.n_vars}" if adata.raw else "  .raw : None")
print(f"  .layers       : {list(adata.layers.keys())}")
print(f"  .obsm keys    : {list(adata.obsm.keys())}")
adata.write_h5ad(out_h5ad, compression="gzip", compression_opts=9)
print(f"Saved: {out_h5ad}")

## 11. Final Summary

In [ ]:
print("="*70)
print("PIPELINE COMPLETE (v1.0)")
print("="*70)

print(f"\nOutput h5ad      : {out_h5ad}")
print(f"scANVI model     : {scanvi_model_dir_new}/")
print(f"Subcluster markers: {marker_dir}/")
print(f"Figures          : {fig_dir}/")

print(f"\nNew annotation column: '{NEW_LABELS_KEY}'")
print(f"Prediction column    : 'scanvi_pred_refined'")
print(f"Latent space         : 'X_scanvi_refined'")
print(f"UMAP                 : 'umap_refined' (from X_scanvi_refined)")

print(f"\nLabel distribution:")
print(adata.obs[NEW_LABELS_KEY].value_counts())

print(f"\nscArches query checklist for future query scripts:")
print(f"  1. labels_key = '{NEW_LABELS_KEY}'")
print(f"  2. scanvi_model_dir = '{scanvi_model_dir_new}'")
print(f"  3. Expected label space: {sorted([c for c in adata.obs[NEW_LABELS_KEY].cat.categories if c != UNLABELED])}")